In [ ]:
import datachain as dc
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

import datachain as dc 
from datachain.sql.functions import path
from transformers import Pipeline, pipeline
from datachain import File

from FantAIno.utils.data_utils import get_secret

# Loading in a Multi-Modal Dataset from S3 with DataChain

In [ ]:
image_df = (dc.read_storage(
    r"s3://fantaino-bucket-085777795487-us-east-2-an/album_art\*",
    type="image", 
    client_config = {
        "key": get_secret("AWS_ACCESS_KEY_ID"),
        "secret": get_secret("AWS_SECRET_ACCESS_KEY")
    }
)
    .settings(cache=True)
    .map(path=lambda file: file.path, output=str)
    .persist()
)

In [ ]:
image_df.show(3)

In [ ]:
lyrics_df = (dc.read_storage(
    r"s3://fantaino-bucket-085777795487-us-east-2-an/lyrics\*",
    client_config = {
        "key": get_secret("AWS_ACCESS_KEY_ID"),
        "secret": get_secret("AWS_SECRET_ACCESS_KEY")
    }
)
    .settings(cache=True)
    .map(path=lambda file: file.path, output=str)
    .persist()
)

In [ ]:
lyrics_df.show(3)

In [ ]:
dc_catalog = (
    image_df
    .settings(cache=True)
    .merge(lyrics_df, on=path.name(image_df.c("file.path")), right_on="filename")
    .persist()
)

In [ ]:
import matplotlib.pyplot as plt
from textwrap import wrap

count = chain.count()
_, axes = plt.subplots(1, count, figsize=(15, 5))

for ax, (img_file, caption) in zip(axes, chain.to_iter("file", "scene")):
    ax.imshow(img_file.read(), cmap="gray")
    ax.axis("off")
    wrapped_caption = "\n".join(wrap(caption.strip(), 40))
    ax.set_title(wrapped_caption, fontsize=10, pad=20)

plt.tight_layout()
plt.show()